# Group Project (Part 1 + Part 2): DQR and DQP

This notebook prepares:
1. **Part 1**: Data Quality Report (DQR) for `ppr-group-22312913-train.csv`
2. **Part 2**: Data Quality Plan (DQP) and implementation of cleaning decisions

The workflow and format follows Week 3/4 labs: setup, initial checks, feature types, tables, visualisations, plan, implementation, and validation.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.backends.backend_pdf import PdfPages

try:
    import seaborn as sns
    sns.set_style('whitegrid')
    USE_SEABORN = True
except Exception:
    USE_SEABORN = False

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)


## Step 1: Load Datasets and Basic Inspection
Use training data for DQR/DQP. Keep test data aside for later project parts.


In [ ]:
train_path = 'ppr-group-22312913-train.csv'
test_path = 'ppr-group-22312913-test.csv'

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print('Train shape:', df_train.shape)
print('Test shape :', df_test.shape)

df_train.head()


## Step 2: Initial Data Checks
Following the lab pattern: dimensions, dtypes, duplicates, constants, and missingness.


In [ ]:
print('Train info:')
print(df_train.info())

print('
Duplicate rows in raw training data:', df_train.duplicated().sum())

constant_columns = [c for c in df_train.columns if df_train[c].nunique(dropna=False) <= 1]
print('Constant columns in raw training data:', constant_columns)

missing_summary = pd.DataFrame({
    'Missing Count': df_train.isna().sum(),
    'Missing %': (df_train.isna().sum() / len(df_train) * 100).round(2)
}).sort_values('Missing %', ascending=False)

missing_summary


## Step 3: Helper Functions (Lab Style)


In [ ]:
def numeric_summary_table(df, numeric_cols):
    if len(numeric_cols) == 0:
        return pd.DataFrame()
    return df[numeric_cols].describe().T


def categorical_summary_table(df, categorical_cols):
    if len(categorical_cols) == 0:
        return pd.DataFrame()
    return df[categorical_cols].describe().T


def data_quality_overview(df):
    overview = pd.DataFrame({
        'Data Type': df.dtypes.astype(str),
        'Missing Count': df.isna().sum(),
        'Missing %': (df.isna().sum() / len(df) * 100).round(2),
        'Unique Count': df.nunique(dropna=False)
    })
    return overview.sort_values(['Missing %', 'Unique Count'], ascending=[False, False])


def plot_numeric_distributions(df, numeric_cols, pdf_path=None):
    if len(numeric_cols) == 0:
        return

    pp = PdfPages(pdf_path) if pdf_path else None

    for col in numeric_cols:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))

        clean_series = df[col].dropna()
        axes[0].hist(clean_series, bins=30, color='steelblue', alpha=0.8)
        axes[0].set_title(f'Histogram: {col}')

        axes[1].boxplot(clean_series, vert=False)
        axes[1].set_title(f'Boxplot: {col}')

        plt.tight_layout()
        if pp:
            pp.savefig(fig)
            plt.close(fig)
        else:
            plt.show()

    if pp:
        pp.close()


def plot_categorical_distributions(df, categorical_cols, top_n=20, pdf_path=None):
    if len(categorical_cols) == 0:
        return

    pp = PdfPages(pdf_path) if pdf_path else None

    for col in categorical_cols:
        vc = df[col].astype('object').value_counts(dropna=False)
        vc = vc.head(top_n)

        fig, ax = plt.subplots(figsize=(13, 4))
        vc.plot(kind='bar', ax=ax, color='teal')
        ax.set_title(f'Bar plot (top {top_n}): {col}')
        ax.set_ylabel('Count')
        ax.tick_params(axis='x', rotation=60)
        plt.tight_layout()

        if pp:
            pp.savefig(fig)
            plt.close(fig)
        else:
            plt.show()

    if pp:
        pp.close()


## Step 4: Preliminary Feature Type Conversion for DQR
As in the labs, we first convert obvious fields to useful analysis types before producing DQR tables/plots.


In [ ]:
df_train_dqr = df_train.copy()

# Convert date
raw_date_col = 'Date of Sale (dd/mm/yyyy)'
df_train_dqr[raw_date_col] = pd.to_datetime(df_train_dqr[raw_date_col], format='%d/%m/%Y', errors='coerce')

# Convert price to numeric euro values
price_col = 'Price (€)'
df_train_dqr[price_col] = (
    df_train_dqr[price_col]
    .astype(str)
    .str.replace('€', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df_train_dqr[price_col] = pd.to_numeric(df_train_dqr[price_col], errors='coerce')

# Convert low-cardinality text fields to category
category_cols = [
    'County',
    'Eircode',
    'Not Full Market Price',
    'VAT Exclusive',
    'Description of Property',
    'Property Size Description'
]

for c in category_cols:
    df_train_dqr[c] = df_train_dqr[c].astype('category')

# Keep high-cardinality Address as object
df_train_dqr['Address'] = df_train_dqr['Address'].astype(str).str.strip()

df_train_dqr.dtypes


## Step 5: Remove Duplicate Rows and Save Cleaned (DQR) Dataset


In [ ]:
before_rows = len(df_train_dqr)
df_train_dqr = df_train_dqr.drop_duplicates().copy()
after_rows = len(df_train_dqr)

print('Rows before duplicate removal:', before_rows)
print('Rows after duplicate removal :', after_rows)
print('Rows removed                 :', before_rows - after_rows)

constant_columns_after = [c for c in df_train_dqr.columns if df_train_dqr[c].nunique(dropna=False) <= 1]
print('Constant columns after duplicate removal:', constant_columns_after)

cleaned_dqr_path = 'ppr-group-22312913-train-cleaned.csv'
df_train_dqr.to_csv(cleaned_dqr_path, index=False)
print('Saved:', cleaned_dqr_path)


## Step 6: Data Quality Report Tables
Numeric and categorical descriptive tables, plus a full overview table.


In [ ]:
numeric_cols = df_train_dqr.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df_train_dqr.select_dtypes(include=['category', 'object']).columns.tolist()

print('Numeric columns:', numeric_cols)
print('Categorical columns:', categorical_cols)


In [ ]:
dqr_overview = data_quality_overview(df_train_dqr)

numeric_stats = numeric_summary_table(df_train_dqr, numeric_cols)
categorical_stats = categorical_summary_table(df_train_dqr, categorical_cols)

print('Data quality overview:')
display(dqr_overview)

print('
Numeric feature summary:')
display(numeric_stats)

print('
Categorical feature summary:')
display(categorical_stats)


## Step 7: Data Quality Report Visualisations


In [ ]:
plot_numeric_distributions(
    df_train_dqr,
    numeric_cols,
    pdf_path='ppr-group-22312913-DQR-numeric-plots.pdf'
)

plot_categorical_distributions(
    df_train_dqr,
    categorical_cols,
    top_n=20,
    pdf_path='ppr-group-22312913-DQR-categorical-plots.pdf'
)

print('Saved DQR plots:')
print('- ppr-group-22312913-DQR-numeric-plots.pdf')
print('- ppr-group-22312913-DQR-categorical-plots.pdf')


In [ ]:
monthly_counts = (
    df_train_dqr.set_index('Date of Sale (dd/mm/yyyy)')
    .resample('ME')
    .size()
)

plt.figure(figsize=(12, 4))
monthly_counts.plot(color='darkorange')
plt.title('Monthly Number of Sales (Train Set)')
plt.xlabel('Month')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


## Step 8: DQR Findings (Summary)

Key findings from the training data analysis:
- The raw training set has duplicate rows that should be removed.
- `Eircode` has high missingness (major data completeness issue).
- `Property Size Description` has very high missingness and inconsistent labels (`greater than 125...` vs `greater than or equal to 125...`).
- `Price (€)` and `Date of Sale (dd/mm/yyyy)` are initially stored as text and must be converted to numeric/datetime for analytics.
- `Address` is high-cardinality free text, so it should be retained as text for now and handled carefully in later modeling/feature engineering parts.


## Part 2: Data Quality Plan (DQP)
Create a feature-by-feature quality plan, justify selected actions, apply them, and validate that no missing values remain.


In [ ]:
dqp_plan = pd.DataFrame([
    {
        'Feature': 'Date of Sale (dd/mm/yyyy)',
        'Final Type': 'datetime64[ns]',
        'Issue(s)': 'Stored as string in raw CSV.',
        'Candidate Solutions': 'Keep as text / parse to datetime.',
        'Selected Action': 'Parse using format %d/%m/%Y with errors=coerce.'
    },
    {
        'Feature': 'Address',
        'Final Type': 'object',
        'Issue(s)': 'High-cardinality free-text; possible spacing inconsistencies.',
        'Candidate Solutions': 'Drop / keep raw / basic text normalisation.',
        'Selected Action': 'Keep and apply strip() whitespace cleanup.'
    },
    {
        'Feature': 'County',
        'Final Type': 'category',
        'Issue(s)': 'Categorical text feature.',
        'Candidate Solutions': 'Object / category.',
        'Selected Action': 'Trim spaces and cast to category.'
    },
    {
        'Feature': 'Eircode',
        'Final Type': 'category',
        'Issue(s)': 'Large proportion of missing values.',
        'Candidate Solutions': 'Drop feature / impute mode / impute Unknown.',
        'Selected Action': 'Impute missing values with Unknown and cast to category.'
    },
    {
        'Feature': 'Price (€)',
        'Final Type': 'float64',
        'Issue(s)': 'Currency symbols/commas in text format.',
        'Candidate Solutions': 'Keep string / parse to numeric euro value.',
        'Selected Action': 'Remove symbols and parse to float.'
    },
    {
        'Feature': 'Not Full Market Price',
        'Final Type': 'category',
        'Issue(s)': 'Binary categorical stored as object.',
        'Candidate Solutions': 'Object / category.',
        'Selected Action': 'Standardise text and cast to category.'
    },
    {
        'Feature': 'VAT Exclusive',
        'Final Type': 'category',
        'Issue(s)': 'Binary categorical stored as object.',
        'Candidate Solutions': 'Object / category.',
        'Selected Action': 'Standardise text and cast to category.'
    },
    {
        'Feature': 'Description of Property',
        'Final Type': 'category',
        'Issue(s)': 'Categorical text; rare label variants.',
        'Candidate Solutions': 'Keep as is / map rare values to Other.',
        'Selected Action': 'Keep values, trim spaces, cast to category.'
    },
    {
        'Feature': 'Property Size Description',
        'Final Type': 'category',
        'Issue(s)': 'Very high missingness; inconsistent category wording.',
        'Candidate Solutions': 'Drop feature / impute Unknown + harmonise categories.',
        'Selected Action': 'Impute Unknown; harmonise 125 sq m category labels; cast to category.'
    }
])

display(dqp_plan)


## Step 9: Implement the Data Quality Plan


In [ ]:
df_train_final = df_train_dqr.copy()

text_cols = ['Address', 'County', 'Eircode', 'Not Full Market Price', 'VAT Exclusive',
             'Description of Property', 'Property Size Description']
for col in text_cols:
    df_train_final[col] = df_train_final[col].astype('object')
    df_train_final[col] = df_train_final[col].where(df_train_final[col].notna(), np.nan)
    df_train_final[col] = df_train_final[col].astype('string').str.strip()

for col in ['Eircode', 'Property Size Description']:
    df_train_final[col] = df_train_final[col].fillna('Unknown')

# Harmonise property size labels
size_col = 'Property Size Description'
df_train_final[size_col] = df_train_final[size_col].replace({
    'greater than 125 sq metres': 'greater than or equal to 125 sq metres'
})

# Defensive date parse fallback
date_col = 'Date of Sale (dd/mm/yyyy)'
if not np.issubdtype(df_train_final[date_col].dtype, np.datetime64):
    df_train_final[date_col] = pd.to_datetime(df_train_final[date_col], format='%d/%m/%Y', errors='coerce')

df_train_final['Price (€)'] = pd.to_numeric(df_train_final['Price (€)'], errors='coerce')

if df_train_final['Price (€)'].isna().any():
    df_train_final['Price (€)'] = df_train_final['Price (€)'].fillna(df_train_final['Price (€)'].median())

if df_train_final[date_col].isna().any():
    date_mode = df_train_final[date_col].mode(dropna=True)
    fallback_date = date_mode.iloc[0] if len(date_mode) > 0 else pd.Timestamp('2016-01-01')
    df_train_final[date_col] = df_train_final[date_col].fillna(fallback_date)

for col in ['County', 'Eircode', 'Not Full Market Price', 'VAT Exclusive',
            'Description of Property', 'Property Size Description']:
    df_train_final[col] = df_train_final[col].astype('category')

df_train_final['Address'] = df_train_final['Address'].fillna('Unknown').astype('object')

# Final safety: ensure no NaN values
for col in df_train_final.columns:
    if str(df_train_final[col].dtype).startswith('category'):
        if df_train_final[col].isna().any():
            df_train_final[col] = df_train_final[col].cat.add_categories(['Unknown']).fillna('Unknown')
    elif df_train_final[col].dtype == 'object':
        df_train_final[col] = df_train_final[col].fillna('Unknown')
    elif np.issubdtype(df_train_final[col].dtype, np.number):
        df_train_final[col] = df_train_final[col].fillna(df_train_final[col].median())
    elif np.issubdtype(df_train_final[col].dtype, np.datetime64):
        df_train_final[col] = df_train_final[col].fillna(df_train_final[col].mode(dropna=True).iloc[0])

print('DQP implementation complete.')


## Step 10: Validation After DQP Cleaning


In [ ]:
print('Final shape:', df_train_final.shape)
print('Duplicate rows:', df_train_final.duplicated().sum())

final_missing = df_train_final.isna().sum().sort_values(ascending=False)
print('
Missing values by column:')
print(final_missing)
print('
Total missing values:', int(final_missing.sum()))

print('
Final dtypes:')
print(df_train_final.dtypes)

df_train_final.head()


In [ ]:
print('Final numeric summary:')
display(df_train_final.select_dtypes(include=['int64', 'float64']).describe().T)

print('Final categorical summary:')
display(df_train_final.select_dtypes(include=['category', 'object']).describe().T)


## Step 11: Save Final Clean Dataset for Next Project Parts


In [ ]:
final_path = 'ppr-group-22312913-train-clean-final.csv'

df_export = df_train_final.copy()
df_export['Date of Sale (dd/mm/yyyy)'] = df_export['Date of Sale (dd/mm/yyyy)'].dt.strftime('%Y-%m-%d')

df_export.to_csv(final_path, index=False)
print('Saved:', final_path)


## Deliverables Generated by This Notebook
- `ppr-group-22312913-train-cleaned.csv` (Part 1 cleaned after duplicate removal/type fixes)
- `ppr-group-22312913-train-clean-final.csv` (Part 2 final cleaned dataset, no missing values)
- `ppr-group-22312913-DQR-numeric-plots.pdf`
- `ppr-group-22312913-DQR-categorical-plots.pdf`

Use `ppr-group-22312913-train-clean-final.csv` as input for Parts 3-5.
